In [1]:
import os
import pandas as pd
import shutil
import unicodedata
import re

# Diretórios
origem = r"C:\Users\wilso\MBA_EMPREENDEDORISMO\3AGD\A3_LOCAL\data"
destino = os.path.join(origem, "md_normalizados")
os.makedirs(destino, exist_ok=True)

# Função para normalizar nome de arquivo
def normalizar_nome(nome):
    # Remover acentos e caracteres especiais
    nfkd = unicodedata.normalize('NFKD', nome).encode('ASCII', 'ignore').decode('ASCII')
    nfkd = re.sub(r'[^A-Za-z0-9._\- ]+', '', nfkd)  # manter apenas caracteres seguros
    nfkd = re.sub(r'\s+', '_', nfkd)  # substituir espaços por _
    return nfkd

# Inventário para log
log = []

# Percorrer recursivamente e processar arquivos .md
for raiz, _, arquivos in os.walk(origem):
    for f in arquivos:
        if f.lower().endswith(".md") and "md_normalizados" not in raiz:
            caminho_origem = os.path.join(raiz, f)
            nome_normalizado = normalizar_nome(f)
            caminho_destino = os.path.join(destino, nome_normalizado)

            try:
                # Testar leitura UTF-8
                with open(caminho_origem, "r", encoding="utf-8") as arq:
                    conteudo = arq.read()

                # Salvar cópia normalizada
                with open(caminho_destino, "w", encoding="utf-8") as arq_out:
                    arq_out.write(conteudo)

                log.append([f, nome_normalizado, caminho_origem, caminho_destino, "OK"])

            except Exception as e:
                log.append([f, f, caminho_origem, "", f"ERRO: {e}"])

# Salvar log
log_csv = os.path.join(r"C:\Users\wilso\MBA_EMPREENDEDORISMO\3AGD\A3_LOCAL\desenvolvimento", "log_normalizacao_md.csv")
pd.DataFrame(log, columns=["arquivo_original", "arquivo_normalizado", "origem", "destino", "status"]).to_csv(log_csv, index=False, encoding="utf-8-sig")

print(f"Normalização concluída. Arquivos limpos em: {destino}")
print(f"Log salvo em: {log_csv}")


Normalização concluída. Arquivos limpos em: C:\Users\wilso\MBA_EMPREENDEDORISMO\3AGD\A3_LOCAL\data\md_normalizados
Log salvo em: C:\Users\wilso\MBA_EMPREENDEDORISMO\3AGD\A3_LOCAL\desenvolvimento\log_normalizacao_md.csv


In [2]:
import os
import pandas as pd
import shutil
import unicodedata
import re
import hashlib

# Diretórios
origem = r"C:\Users\wilso\MBA_EMPREENDEDORISMO\3AGD\A3_LOCAL\data"
destino = os.path.join(origem, "md_normalizados_v2")
os.makedirs(destino, exist_ok=True)

# Função para normalizar nome de arquivo
def normalizar_nome(nome):
    nfkd = unicodedata.normalize('NFKD', nome).encode('ASCII', 'ignore').decode('ASCII')
    nfkd = re.sub(r'[^A-Za-z0-9._\- ]+', '', nfkd)  # apenas caracteres seguros
    nfkd = re.sub(r'\s+', '_', nfkd)                # espaços -> _
    return nfkd

# Função para calcular hash do conteúdo
def calcular_hash(caminho):
    hasher = hashlib.sha1()
    with open(caminho, "rb") as f:
        while chunk := f.read(8192):
            hasher.update(chunk)
    return hasher.hexdigest()

# Inventário para log
log = []
hash_registro = {}  # Mapeia nome -> lista de hashes já salvos

# Percorrer recursivamente e processar arquivos .md
for raiz, _, arquivos in os.walk(origem):
    for f in arquivos:
        if f.lower().endswith(".md") and "md_normalizados" not in raiz:
            caminho_origem = os.path.join(raiz, f)
            nome_base = normalizar_nome(os.path.splitext(f)[0]) + ".md"
            hash_arquivo = calcular_hash(caminho_origem)

            # Verifica se já existe arquivo com este nome
            if nome_base not in hash_registro:
                hash_registro[nome_base] = [hash_arquivo]
                destino_final = os.path.join(destino, nome_base)
            else:
                # Se o hash é diferente, cria versão com prefixo do hash
                if hash_arquivo not in hash_registro[nome_base]:
                    hash_registro[nome_base].append(hash_arquivo)
                    destino_final = os.path.join(destino, f"{os.path.splitext(nome_base)[0]}_{hash_arquivo[:8]}.md")
                else:
                    # Arquivo idêntico já foi salvo, não copiar de novo
                    log.append([f, nome_base, caminho_origem, "", "DUPLICADO - IGUAL"])
                    continue

            # Copiar arquivo
            shutil.copy2(caminho_origem, destino_final)
            log.append([f, os.path.basename(destino_final), caminho_origem, destino_final, "OK"])

# Salvar log
log_csv = os.path.join(r"C:\Users\wilso\MBA_EMPREENDEDORISMO\3AGD\A3_LOCAL\desenvolvimento", "log_normalizacao_md_v2.csv")
pd.DataFrame(log, columns=["arquivo_original", "arquivo_final", "origem", "destino", "status"]).to_csv(log_csv, index=False, encoding="utf-8-sig")

print(f"Reprocessamento concluído. Arquivos limpos em: {destino}")
print(f"Log salvo em: {log_csv}")


Reprocessamento concluído. Arquivos limpos em: C:\Users\wilso\MBA_EMPREENDEDORISMO\3AGD\A3_LOCAL\data\md_normalizados_v2
Log salvo em: C:\Users\wilso\MBA_EMPREENDEDORISMO\3AGD\A3_LOCAL\desenvolvimento\log_normalizacao_md_v2.csv


In [3]:
import os
import shutil

base = r"C:\Users\wilso\MBA_EMPREENDEDORISMO\3AGD\A3_LOCAL\data"
nova_ssot = os.path.join(base, "ssot_a3")
legacy = os.path.join(base, "legacy")
os.makedirs(legacy, exist_ok=True)

# 1. Mover a ssot_a3 original para legacy
orig_ssot = os.path.join(base, "ssot_a3")
if os.path.exists(orig_ssot):
    shutil.move(orig_ssot, os.path.join(legacy, "ssot_a3_original"))

# 2. Mover md_normalizados para legacy
md_normalizados = os.path.join(base, "md_normalizados")
if os.path.exists(md_normalizados):
    shutil.move(md_normalizados, os.path.join(legacy, "md_normalizados"))

# 3. Mover chroma antigo para legacy
chroma_antigo = os.path.join(base, "chroma_ssot_a3")
if os.path.exists(chroma_antigo):
    shutil.move(chroma_antigo, os.path.join(legacy, "chroma_ssot_a3_antigo"))

# 4. Renomear md_normalizados_v2 para ssot_a3
md_v2 = os.path.join(base, "md_normalizados_v2")
if os.path.exists(md_v2):
    shutil.move(md_v2, nova_ssot)

print("Nova SSOT estabelecida em:", nova_ssot)
print("Backups movidos para:", legacy)


Nova SSOT estabelecida em: C:\Users\wilso\MBA_EMPREENDEDORISMO\3AGD\A3_LOCAL\data\ssot_a3
Backups movidos para: C:\Users\wilso\MBA_EMPREENDEDORISMO\3AGD\A3_LOCAL\data\legacy
